In [1]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

In [2]:
dagshub.init(repo_owner='gdzag22', repo_name='ml_assignment_1', mlflow=True)

Accessing as gdzag22

Initialized MLflow to track repo "gdzag22/ml_assignment_1"

Repository gdzag22/ml_assignment_1 initialized!

In [3]:
model = mlflow.sklearn.load_model("models:/house_prices_best_model/2")
print("loaded")

loaded


In [4]:
df_test = pd.read_csv('data/test.csv')
print(df_test.shape)
df_test.head()

(1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [5]:
test_ids = df_test['Id']
df_test = df_test.drop(columns=['Id'])

cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
df_test = df_test.drop(columns=cols_to_drop)

print(df_test.shape)

(1459, 73)


In [6]:
cat_cols_test = [col for col in df_test.columns if df_test[col].dtype == 'object']
num_cols_test = [col for col in df_test.columns if df_test[col].dtype != 'object']

for col in num_cols_test:
    df_test[col] = df_test[col].fillna(df_test[col].median())

for col in cat_cols_test:
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

print(df_test.isna().sum().sum())

0


In [7]:
df_test_encoded = pd.get_dummies(df_test, columns=cat_cols_test, drop_first=True)
print(df_test_encoded.shape)

(1459, 213)


In [8]:
import json
with open('data/train_columns.json', 'r') as f:
    train_columns = json.load(f)

df_test_encoded = df_test_encoded.reindex(columns=train_columns, fill_value=0)
df_test_encoded = df_test_encoded.astype(int)

print(df_test_encoded.shape)

(1459, 227)


In [9]:
with open('data/selected_features_dt.json', 'r') as f:
    selected_features_dt = json.load(f)

df_test_final = df_test_encoded[selected_features_dt]
print(df_test_final.shape)

(1459, 100)


In [10]:
predictions = model.predict(df_test_final)
predictions = np.expm1(predictions)

print(predictions[:5])

[124841.40455583 155045.67695355 181911.76495126 184450.67768165
 200166.87721036]


In [11]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv('data/submission.csv', index=False)
print(submission.shape)
submission.head()

(1459, 2)


,Id,SalePrice
0,1461,124841.404556
1,1462,155045.676954
2,1463,181911.764951
3,1464,184450.677682
4,1465,200166.877210
